**Import libraries**

In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [3]:
# Load cleaned dataset

In [4]:
df_model = pd.read_csv("../data_process/data_clean_2.csv")
df_model["MthCalDt"] = pd.to_datetime(df_model["MthCalDt"])

print("Shape:", df_model.shape)
print("Date range:", df_model["MthCalDt"].min(), "to", df_model["MthCalDt"].max())

Shape: (1501178, 24)
Date range: 1998-04-30 00:00:00 to 2025-11-28 00:00:00


In [5]:
# Define features and targets

feature_cols = [
    "Mkt-RF",
    "SMB",
    "HML",
    "RMW",
    "CMA",
    "ret_lag1",
    "ret_lag2",
    "ret_lag3",
    "log_size",
]

target_col = "y_next"

months = sorted(df_model["DateKey"].dropna().unique())

print("Total months:", len(months))
print("First months:", months[:5])
print("Last months:", months[-5:])

Total months: 332
First months: [np.int64(199804), np.int64(199805), np.int64(199806), np.int64(199807), np.int64(199808)]
Last months: [np.int64(202507), np.int64(202508), np.int64(202509), np.int64(202510), np.int64(202511)]


In [6]:
print(len(months))

332


In [7]:
# Run rolling 36-month Forest

window_size = 36
results = []

for i in range(window_size, len(months)):

    test_month = months[i]
    train_months = months[i - window_size : i]

    train_data = df_model[df_model["DateKey"].isin(train_months)]
    test_data = df_model[df_model["DateKey"] == test_month]

    if len(train_data) == 0 or len(test_data) == 0:
        print("Skipping:", test_month)
        continue

    X_train = train_data[feature_cols]
    y_train = train_data[target_col]
    X_test = test_data[feature_cols]

    model = RandomForestRegressor(
        n_estimators=50,
        max_depth=5,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    temp = test_data.copy()
    temp["predicted_return"] = preds

    results.append(temp)

pred_df = pd.concat(results, ignore_index=True)

# KEEP ONLY POST-2022
pred_df = pred_df[pred_df["MthCalDt"] >= "2022-01-01"].copy()

print("RF Prediction shape:", pred_df.shape)
print("RF months:", pred_df["DateKey"].nunique())
display(pred_df.head())

RF Prediction shape: (252411, 25)
RF months: 47


,PERMNO,HdrCUSIP,Ticker,PERMCO,MthCalDt,MthPrc,MthCap,MthRet,ShrOut,MthPrc_Abs,DateKey,Mkt-RF,SMB,HML,RMW,CMA,RF,ExcessRet,y_next,ret_lag1,ret_lag2,ret_lag3,history_len,log_size,predicted_return
1098683,10026,46603210,JJSF,7976,2022-01-31,151.690,2898795.90,-0.039694,19110,151.690,202201,-0.0615,-0.0401,0.1286,0.0077,0.0773,0.0,-0.039694,0.079306,0.161156,-0.074348,-0.034485,321,14.879806,0.039448
1098684,10032,72913210,PLXS,7980,2022-01-31,77.520,2170327.44,-0.191574,27997,77.520,202201,-0.0615,-0.0401,0.1286,0.0077,0.0773,0.0,-0.191574,0.050955,0.139548,-0.036418,-0.023375,321,14.590389,0.039448
1098685,10044,77467X10,RMCF,7992,2022-01-31,7.912,48896.16,0.007898,6180,7.912,202201,-0.0615,-0.0401,0.1286,0.0077,0.0773,0.0,0.007898,-0.007836,-0.076571,-0.024110,0.177027,286,10.797454,0.043917
1098686,10051,41043F20,HNGR,7999,2022-01-31,18.130,701794.17,0.000000,38709,18.130,202201,-0.0615,-0.0401,0.1286,0.0077,0.0773,0.0,0.000000,-0.000552,0.074589,-0.096895,-0.149362,255,13.461395,0.038864
1098687,10065,00621210,ADX,20023,2022-01-31,18.440,2047337.88,-0.049974,111027,18.440,202201,-0.0615,-0.0401,0.1286,0.0077,0.0773,0.0,-0.049974,-0.029364,0.029062,0.019589,0.071212,321,14.532051,0.050495


In [8]:
pred_df["rank"] = pred_df.groupby("DateKey")["predicted_return"].rank(
    method="first", ascending=False
)

top6_rf = pred_df[pred_df["rank"] <= 6].copy()

print("RF Top6 shape:", top6_rf.shape)
print("RF months:", top6_rf["DateKey"].nunique())

display(
    top6_rf[["DateKey", "Ticker", "predicted_return", "rank"]]
    .sort_values(["DateKey", "rank"])
    .head(12)
)

RF Top6 shape: (282, 26)
RF months: 47


,DateKey,Ticker,predicted_return,rank
1098698,202201,INDB,0.050807,1.0
1098710,202201,AMWD,0.050807,2.0
1098712,202201,ADM,0.050807,3.0
1098732,202201,CERN,0.050807,4.0
1098734,202201,MKL,0.050807,5.0
1098747,202201,FBNC,0.050807,6.0
1103905,202202,DLHC,0.086181,1.0
1103921,202202,CLFD,0.086181,2.0
1103938,202202,CERN,0.086181,3.0
1103947,202202,PLAB,0.086181,4.0


In [12]:
# Evaluate Random Forest performance
mse = mean_squared_error(pred_df["y_next"], pred_df["predicted_return"])
mae = mean_absolute_error(pred_df["y_next"], pred_df["predicted_return"])

print("RF Performance (Post-2022)")
print("MSE:", mse)
print("MAE:", mae)

RF Performance (Post-2022)
MSE: 0.011011818260331712
MAE: 0.07445656038587409


In [13]:
# Inspect prediction distribution

pred_df["predicted_return"].describe()

count    252411.000000
mean          0.005592
std           0.032821
min          -0.085446
25%          -0.013833
50%           0.000794
75%           0.025522
max           0.165119
Name: predicted_return, dtype: float64

In [14]:
# Save top 6 Random Forest result

top6_rf.to_csv("../data/rf_top6_by_month_post2022.csv", index=False)

print("Saved RF top 6 results")

Saved RF top 6 results


In [15]:
top6_ols = pd.read_csv("../data/ols_top6_by_month_post2022.csv")
top6_rf = pd.read_csv("../data/rf_top6_by_month_post2022.csv")

ols_portfolio = top6_ols.groupby("DateKey")["y_next"].mean().reset_index()
rf_portfolio = top6_rf.groupby("DateKey")["y_next"].mean().reset_index()

ols_portfolio.rename(columns={"y_next": "ols_return"}, inplace=True)
rf_portfolio.rename(columns={"y_next": "rf_return"}, inplace=True)

portfolio_compare = pd.merge(ols_portfolio, rf_portfolio, on="DateKey")

print("OLS avg return:", portfolio_compare["ols_return"].mean())
print("RF avg return:", portfolio_compare["rf_return"].mean())

print("OLS std:", portfolio_compare["ols_return"].std())
print("RF std:", portfolio_compare["rf_return"].std())

OLS avg return: 0.0063859636524822684
RF avg return: 0.0183672914893617
OLS std: 0.09074270896135214
RF std: 0.052718158631642455


In [16]:
print(top6_ols["y_next"].describe())
print(top6_rf["y_next"].describe())

count    282.000000
mean       0.006386
std        0.181910
min       -0.302113
25%       -0.112439
50%       -0.011135
75%        0.114841
max        0.442129
Name: y_next, dtype: float64
count    282.000000
mean       0.018367
std        0.117564
min       -0.302113
25%       -0.047368
50%        0.014014
75%        0.067801
max        0.442129
Name: y_next, dtype: float64


In [17]:
print("OLS months:", top6_ols["DateKey"].nunique())
print("RF months:", top6_rf["DateKey"].nunique())

print("\nOLS picks per month:")
print(top6_ols.groupby("DateKey").size().value_counts())

print("\nRF picks per month:")
print(top6_rf.groupby("DateKey").size().value_counts())

OLS months: 47
RF months: 47

OLS picks per month:
6    47
Name: count, dtype: int64

RF picks per month:
6    47
Name: count, dtype: int64


In [22]:
# Baseline: equal-weight all stocks per month
baseline = pred_df.groupby("DateKey")["y_next"].mean().reset_index()
baseline.rename(columns={"y_next": "baseline_return"}, inplace=True)

# Merge with your portfolios
compare_all = pd.merge(portfolio_compare, baseline, on="DateKey")

print("Baseline avg return:", compare_all["baseline_return"].mean())
print("Baseline std:", compare_all["baseline_return"].std())

Baseline avg return: 0.005629162277260785
Baseline std: 0.042234249409725776


In [23]:
# =========================
# Summary comparison
# =========================

import numpy as np

# Averages
ols_avg = portfolio_compare["ols_return"].mean()
rf_avg = portfolio_compare["rf_return"].mean()
base_avg = compare_all["baseline_return"].mean()

# Risk (std)
ols_std = portfolio_compare["ols_return"].std()
rf_std = portfolio_compare["rf_return"].std()
base_std = compare_all["baseline_return"].std()

# Sharpe (simple, no RF subtraction)
ols_sharpe = ols_avg / ols_std
rf_sharpe = rf_avg / rf_std
base_sharpe = base_avg / base_std

print("=== PERFORMANCE SUMMARY ===\n")

print("Average Return:")
print(f"Baseline: {base_avg:.4f}")
print(f"OLS:      {ols_avg:.4f}")
print(f"RF:       {rf_avg:.4f}\n")

print("Risk (Std Dev):")
print(f"Baseline: {base_std:.4f}")
print(f"OLS:      {ols_std:.4f}")
print(f"RF:       {rf_std:.4f}\n")

print("Sharpe Ratio:")
print(f"Baseline: {base_sharpe:.4f}")
print(f"OLS:      {ols_sharpe:.4f}")
print(f"RF:       {rf_sharpe:.4f}\n")

# Improvements
rf_vs_base = (rf_avg - base_avg) / abs(base_avg)
rf_vs_ols = (rf_avg - ols_avg) / abs(ols_avg)

print("=== IMPROVEMENT ===\n")
print(f"RF vs Baseline return improvement: {rf_vs_base:.2%}")
print(f"RF vs OLS return improvement:      {rf_vs_ols:.2%}")

=== PERFORMANCE SUMMARY ===

Average Return:
Baseline: 0.0056
OLS:      0.0064
RF:       0.0184

Risk (Std Dev):
Baseline: 0.0422
OLS:      0.0907
RF:       0.0527

Sharpe Ratio:
Baseline: 0.1333
OLS:      0.0704
RF:       0.3484

=== IMPROVEMENT ===

RF vs Baseline return improvement: 226.29%
RF vs OLS return improvement:      187.62%


In [19]:
top6_rf.groupby("Ticker").size().sort_values(ascending=False)

Ticker
JJSF    30
ADX     29
PLXS    29
ORCL    28
MSFT    20
        ..
ERIC     1
ENZL     1
ENTA     1
ELD      1
ZGNX     1
Length: 99, dtype: int64